# Ejercicio · Comparación de Técnicas de Preprocesamiento

## Contexto

Trabajas como científico/a de datos en un equipo que necesita predecir si
una persona gana **más de 50K USD al año** a partir de datos socio-demográficos
(dataset *Adult Census Income* de UCI). El equipo de modelado ya tiene en
mente usar **Logistic Regression** y **Random Forest**, pero el debate
está abierto sobre **cómo preprocesar las variables**.

Tu tarea: comparar varias estrategias y entregar una recomendación
**basada en datos**, no en intuición.

## Reglas del Juego

1. Usa el **mismo split** train/test para todas las estrategias
   (semilla fija → comparaciones justas).
2. Usa **las mismas dos clases de modelo** (Logistic Regression y Random
   Forest) con hiperparámetros por defecto (no es un ejercicio de tuning).
3. Reporta **Accuracy, F1 y ROC-AUC** en el conjunto de prueba.
4. **Documenta tus decisiones**: cada estrategia debe tener un párrafo
   que explique qué cambia y por qué.

## Dataset

- **Fuente:** https://archive.ics.uci.edu/dataset/2/adult
- **Filas:** ~48,800
- **Target:** `income` (`>50K` vs `<=50K`)
- **Faltantes:** codificados como `?` (no como `NaN`)
- **Licencia:** CC BY 4.0


## Lo que tienes que hacer

Vas a construir y comparar **6 estrategias de preprocesamiento**, cada una
entrenada con dos modelos (LogReg y RF). Al final, debes producir:

1. Una **tabla comparativa** con las 12 combinaciones (6 estrategias × 2 modelos).
2. Una **gráfica** que muestre el F1 por estrategia y modelo.
3. Una **respuesta justificada** a las preguntas de reflexión.

Las 6 estrategias son:

| # | Nombre | Imputación | Categóricas | Numéricas |
|---|---|---|---|---|
| 1 | `drop_minimal` | Eliminar filas con NaN | OneHot | Sin escalar |
| 2 | `impute_ohe` | Mediana / moda | OneHot | StandardScaler |
| 3 | `impute_ohe_log` | Mediana / moda | OneHot | log1p + StandardScaler |
| 4 | `impute_ohe_minmax` | Mediana / moda | OneHot | MinMaxScaler |
| 5 | `ordinal_all` | Mediana / moda | **Ordinal** | StandardScaler |
| 6 | `drop_high_card` | Mediana / moda | OneHot SIN `native-country` | StandardScaler |


## Paso 1 · Imports y configuración

In [ ]:
import io
import urllib.request
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)

URL_TRAIN = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
URL_TEST = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

COLS = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income",
]

print("Descargando Adult Census Income desde UCI...")
train_raw = urllib.request.urlopen(URL_TRAIN).read().decode("utf-8")
test_raw = urllib.request.urlopen(URL_TEST).read().decode("utf-8")

df_train = pd.read_csv(io.StringIO(train_raw), names=COLS, skipinitialspace=True)
# adult.test trae una primera línea de comentario y un punto final en income
df_test = pd.read_csv(io.StringIO(test_raw), names=COLS,
                      skipinitialspace=True, skiprows=1)
df_test["income"] = df_test["income"].str.rstrip(".")

df = pd.concat([df_train, df_test], ignore_index=True)
print(f"Shape: {df.shape}")
df.head()


## Paso 2 · Diagnóstico inicial

> **TODO:** observa cuántos faltantes hay y la proporción de la clase positiva.
> Anota el porcentaje de filas que perderías si simplemente hicieras `dropna()`.


In [ ]:
# 1. Convertir "?" en NaN explícito (es así como lo trae el dataset original)
df = df.replace("?", np.nan)

print("Faltantes por columna:")
print(df.isna().sum()[df.isna().sum() > 0])

print(f"\nDistribución del target:")
print(df["income"].value_counts(normalize=True).round(3))


## Paso 3 · Split estratificado

In [ ]:
from sklearn.model_selection import train_test_split

y = (df["income"] == ">50K").astype(int)
X = df.drop(columns=["income"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Proporción positivos en train: {y_train.mean():.3f}")
print(f"Proporción positivos en test:  {y_test.mean():.3f}")


## Paso 4 · Agrupar columnas por tipo

In [ ]:
# Agrupamos las columnas por tipo para usarlas con ColumnTransformer
NUMERIC = ["age", "fnlwgt", "capital-gain", "capital-loss", "hours-per-week"]
NUMERIC_ORDINAL = ["education-num"]   # variable ordinal codificada como número
CATEGORICAL_LOW = ["workclass", "marital-status", "occupation",
                   "relationship", "race", "sex"]
CATEGORICAL_HIGH = ["native-country"]  # ~40 valores distintos
CATEGORICAL_EDU = ["education"]        # alternativa ordinal a education-num

print("Numéricas:", NUMERIC)
print("Ordinal numérica:", NUMERIC_ORDINAL)
print("Categóricas baja card.:", CATEGORICAL_LOW)
print("Categórica alta card.:", CATEGORICAL_HIGH)
print("Educación (categórica):", CATEGORICAL_EDU)
print(f"\nNiveles únicos en native-country: {df['native-country'].nunique()}")
print(f"Niveles únicos en education: {df['education'].nunique()}")


## Paso 5 · Helper para evaluar pipelines

Esta función ya está completa: te ahorra escribir el mismo código una y
otra vez. Léela para entender qué hace.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

MODELS = {
    "LogisticRegression": LogisticRegression(max_iter=2000, n_jobs=-1, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
}


def evaluar_pipeline(preprocessor, X_tr, y_tr, X_te, y_te, etiqueta):
    """
    Entrena LogReg y RandomForest con el preprocessor dado y devuelve
    una lista de filas con métricas. Cada fila incluye 'estrategia' y
    'modelo' para poder concatenar luego.
    """
    from sklearn.pipeline import Pipeline

    rows = []
    for nombre_modelo, modelo in MODELS.items():
        pipe = Pipeline([("pre", preprocessor), ("clf", modelo)])
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)
        y_proba = pipe.predict_proba(X_te)[:, 1]
        n_feats = pipe.named_steps["pre"].transform(X_tr.head(5)).shape[1]
        rows.append({
            "estrategia": etiqueta,
            "modelo": nombre_modelo,
            "n_features": n_feats,
            "accuracy": accuracy_score(y_te, y_pred),
            "f1": f1_score(y_te, y_pred),
            "roc_auc": roc_auc_score(y_te, y_proba),
        })
    return rows


## Paso 6 · Construye los 6 preprocesadores

A continuación tienes una plantilla por estrategia. **Completa los `TODO`**
con el preprocesador correcto. Pista: todos usan `ColumnTransformer`
con uno o dos `Pipeline` adentro.

> **Tip importante**: en sklearn 1.2+ usa `OneHotEncoder(sparse_output=False)`
> para devolver arrays densos.


### Estrategia 1 · `drop_minimal`

- Quitar todas las filas con NaN antes de entrenar.
- Codificar TODAS las categóricas con `OneHotEncoder`.
- **No** escalar las numéricas (es la versión "naive").


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Filtrar filas sin NaN ANTES de entrenar
mask_train = X_train.notna().all(axis=1)
mask_test = X_test.notna().all(axis=1)
X_train_drop = X_train[mask_train]
y_train_drop = y_train[mask_train]
X_test_drop = X_test[mask_test]
y_test_drop = y_test[mask_test]
print(f"Tras dropna - Train: {len(X_train_drop):,} (perdiste {len(X_train) - len(X_train_drop):,} filas)")

# TODO: completar el ColumnTransformer
preprocessor_1 = ColumnTransformer([
    # ("num", ..., NUMERIC + NUMERIC_ORDINAL),
    # ("cat", ..., CATEGORICAL_LOW + CATEGORICAL_HIGH + CATEGORICAL_EDU),
], remainder="drop")


### Estrategia 2 · `impute_ohe`

- Imputar numéricas con **mediana**, categóricas con **moda**.
- OneHot para todas las categóricas.
- StandardScaler para las numéricas.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# TODO: construir preprocessor_2
preprocessor_2 = ColumnTransformer([
    # ("num", Pipeline([...]), NUMERIC + NUMERIC_ORDINAL),
    # ("cat", Pipeline([...]), CATEGORICAL_LOW + CATEGORICAL_HIGH + CATEGORICAL_EDU),
], remainder="drop")


### Estrategia 3 · `impute_ohe_log`

- Igual que la anterior, pero antes del escalado aplica `log1p`.
- Pista: usa `FunctionTransformer(np.log1p, validate=False)` o crea una
  función propia que aplique `log(1 + max(0, x))` para evitar negativos.


In [ ]:
from sklearn.preprocessing import FunctionTransformer

def _log_safe(x):
    return np.log1p(np.clip(x, 0, None))

# TODO: construir preprocessor_3
preprocessor_3 = ColumnTransformer([
    # ("num", Pipeline([imputer, log, scaler]), NUMERIC + NUMERIC_ORDINAL),
    # ("cat", Pipeline([imputer, ohe]), CATEGORICAL_LOW + CATEGORICAL_HIGH + CATEGORICAL_EDU),
], remainder="drop")


### Estrategia 4 · `impute_ohe_minmax`

- Igual que la 2, pero con `MinMaxScaler` en lugar de `StandardScaler`.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# TODO: construir preprocessor_4
preprocessor_4 = ColumnTransformer([
    # ...
], remainder="drop")


### Estrategia 5 · `ordinal_all`

- Imputar igual que antes.
- Codificar TODAS las categóricas con **OrdinalEncoder** en lugar de OneHot.
- Conserva StandardScaler en las numéricas.

> ⚠️ Esta estrategia es deliberadamente **mala** para Logistic Regression.
> Vas a observar empíricamente por qué.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# TODO: construir preprocessor_5
preprocessor_5 = ColumnTransformer([
    # ("num", ...),
    # ("cat", Pipeline([imputer, OrdinalEncoder(...)]), ...),
], remainder="drop")


### Estrategia 6 · `drop_high_card`

- Igual que la estrategia 2, pero **eliminando** la variable
  `native-country` (alta cardinalidad).
- Verás cuántas columnas pierdes y si vale la pena.


In [ ]:
# TODO: construir preprocessor_6 (sin CATEGORICAL_HIGH)
preprocessor_6 = ColumnTransformer([
    # ...
], remainder="drop")


## Paso 7 · Ejecutar todas las estrategias

> **TODO:** descomenta este bloque después de completar los 6 preprocesadores.


In [ ]:
# resultados = []
# resultados += evaluar_pipeline(preprocessor_1, X_train_drop, y_train_drop,
#                                X_test_drop, y_test_drop, "drop_minimal")
# resultados += evaluar_pipeline(preprocessor_2, X_train, y_train,
#                                X_test, y_test, "impute_ohe")
# resultados += evaluar_pipeline(preprocessor_3, X_train, y_train,
#                                X_test, y_test, "impute_ohe_log")
# resultados += evaluar_pipeline(preprocessor_4, X_train, y_train,
#                                X_test, y_test, "impute_ohe_minmax")
# resultados += evaluar_pipeline(preprocessor_5, X_train, y_train,
#                                X_test, y_test, "ordinal_all")
# resultados += evaluar_pipeline(preprocessor_6, X_train, y_train,
#                                X_test, y_test, "drop_high_card")

# resultados_df = pd.DataFrame(resultados)
# resultados_df.round(4)


## Paso 8 · Visualizar la comparación

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# for ax, metric in zip(axes, ["accuracy", "f1", "roc_auc"]):
#     sns.barplot(data=resultados_df, x="estrategia", y=metric, hue="modelo", ax=ax)
#     ax.set_title(metric)
#     ax.tick_params(axis="x", rotation=45)
# plt.tight_layout()
# plt.show()


## Paso 9 · Spread por modelo

> **TODO:** calcula, para cada modelo, la diferencia entre el mejor y el
> peor valor de F1 entre todas las estrategias. ¿Qué modelo tiene el
> mayor spread?


In [ ]:
# for modelo in resultados_df["modelo"].unique():
#     sub = resultados_df[resultados_df["modelo"] == modelo]
#     d_acc = sub["accuracy"].max() - sub["accuracy"].min()
#     d_f1 = sub["f1"].max() - sub["f1"].min()
#     d_auc = sub["roc_auc"].max() - sub["roc_auc"].min()
#     print(f"{modelo:20s} Δacc={d_acc:.4f}  Δf1={d_f1:.4f}  Δauc={d_auc:.4f}")


## Preguntas de Reflexión

Responde con tus propias palabras (no hace falta que sean respuestas largas):

1. ¿Cuál es la mejor estrategia para **LogisticRegression** según tu tabla?
   ¿Y para **RandomForest**? ¿Coinciden?
2. ¿Por qué la estrategia `ordinal_all` funciona pésimo en LogReg pero
   bien en RandomForest? Explica con tus palabras qué está pasando.
3. ¿Cuántas columnas pierdes al hacer `dropna()`? ¿Vale la pena en este caso?
4. La estrategia `drop_high_card` reduce mucho la dimensionalidad
   (de ~105 a ~64 features). ¿Mejora o empeora las métricas? ¿Por qué crees?
5. ¿Esperabas que `log1p` mejorara las métricas? ¿Lo hizo? Si no, ¿qué
   crees que pasa con variables como `capital-gain` (que tiene muchos ceros)?
6. Si fueras el/la responsable de poner este modelo en producción, ¿qué
   estrategia escogerías? Justifica con datos, no con intuición.

## Entregable

Comparte con tu profesor:

1. El notebook completado.
2. La **tabla final** con las 12 filas (6 estrategias × 2 modelos).
3. Un **párrafo de 4-6 líneas** explicando tu elección final
   (modelo + estrategia) y por qué.
